# SL-2 - Apprentissage et Connaissance : EBL & RBL (C#)

> **Jumeau C#** du notebook [SL-2-KnowledgeBasedLearning.ipynb](SL-2-KnowledgeBasedLearning.ipynb).
> Version Python ↔ Version C# (.NET Interactive) — parité pédagogique du marathon #4956.
> Algorithmes AIMA ch.19.3 (EBL) + 19.4 (RBL), implémentés from-scratch en C# pur.

## Objectifs d'apprentissage

1. **Distinguer** les 4 contraintes d'apprentissage (induction pure, EBL, RBL, KBIL).
2. **Implémenter** la vérification par énumération de modèles (mini-monde propositionnel).
3. **Exécuter** le chaînage avant avec unification (moteur EBL).
4. **Construire** un arbre de preuve EBL (simplification arithmétique) + le variabiliser.
5. **Mesurer** le speedup apporté par les règles apprises.
6. **Vérifier** une détermination RBL (attributs déterminants).

### Prérequis
- Notions de logique propositionnelle + chaînage avant.
- .NET 9.0 + dotnet-interactive (kernel `.net-csharp`).
- Avoir suivi [SL-1-LogicalLearning-Csharp.ipynb](SL-1-LogicalLearning-Csharp.ipynb) (CBH / Version Space).

### Durée estimée : 55 minutes

### Références
- Russell & Norvig, *AIMA* (3e éd.), §19.3-19.4.
- Notebook Python : [SL-2-KnowledgeBasedLearning.ipynb](SL-2-KnowledgeBasedLearning.ipynb).


## 1. Les 4 contraintes d'apprentissage

AIMA distingue 4 façons dont la connaissance interagit avec l'apprentissage. On les
vérifie par **énumération de modèles** sur un mini-monde propositionnel à 3 atomes
(`pointu`, `perce`, `tue`) — l'exemple du bâton de Zog.

In [1]:
using System.Linq;

// Mini-monde propositionnel : 3 atomes.
string[] ATOMS = {"pointu", "perce", "tue"};

// Une formule = une fonction modèle -> bool. Un modèle = dict atome -> bool.
bool Entails(List<Func<Dictionary<string,bool>,bool>> kb,
             Func<Dictionary<string,bool>,bool> query) {
    // KB |= query ssi query est vraie dans TOUS les modèles où KB est vraie.
    foreach (var values in AllAssignments(ATOMS.Length)) {
        var model = new Dictionary<string,bool>();
        for (int i = 0; i < ATOMS.Length; i++) model[ATOMS[i]] = values[i];
        if (kb.All(f => f(model)) && !query(model)) return false;
    }
    return true;
}

// Énumère les 2^n combinaisons booléennes (produit cartésien).
IEnumerable<bool[]> AllAssignments(int n) {
    for (int mask = 0; mask < (1 << n); mask++) {
        var v = new bool[n];
        for (int i = 0; i < n; i++) v[i] = (mask & (1 << i)) != 0;
        yield return v;
    }
}

// Connaissance de fond : pointu => perce, perce => tue.
var background = new List<Func<Dictionary<string,bool>,bool>> {
    m => (!m["pointu"]) || m["perce"],
    m => (!m["perce"]) || m["tue"],
};
// Observation : Description = pointu ; Classification = tue.
Func<Dictionary<string,bool>,bool> description = m => m["pointu"];
Func<Dictionary<string,bool>,bool> classification = m => m["tue"];
// Hypothèse candidate H : pointu => tue.
Func<Dictionary<string,bool>,bool> hypothesis = m => (!m["pointu"]) || m["tue"];

Console.WriteLine("Contraintes d'apprentissage (AIMA 19.3-19.5) vérifiées par énumération");
Console.WriteLine(new string('=', 70));
Console.WriteLine();
Console.WriteLine("1. Induction pure (SL-1) : H ^ Desc |= Class");
Console.WriteLine($"   -> {Entails(new(){hypothesis, description}, classification)}");
Console.WriteLine();
Console.WriteLine("2. EBL : Background |= Hypothese");
Console.WriteLine($"   (pointu=>perce)^(perce=>tue) |= (pointu=>tue) ? -> {Entails(background, hypothesis)}");
Console.WriteLine("   H était déjà déductible : EBL ne découvre rien, il COMPILE.");
Console.WriteLine();
Console.WriteLine("3. RBL : Background ^ Desc ^ Class |= Hypothese");
var rblKb = new List<Func<Dictionary<string,bool>,bool>>{description, classification};
rblKb.AddRange(background);
Console.WriteLine($"   -> {Entails(rblKb, hypothesis)}");
Console.WriteLine();
Console.WriteLine("4. KBIL : Background ^ H ^ Desc |= Class");
var kbilKb = new List<Func<Dictionary<string,bool>,bool>>{hypothesis, description};
kbilKb.AddRange(background);
Console.WriteLine($"   -> {Entails(kbilKb, classification)}");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Contraintes d'apprentissage (AIMA 19.3-19.5) vérifiées par énumération


1. Induction pure (SL-1) : H ^ Desc |= Class


   -> True


2. EBL : Background |= Hypothese


...

   H était déjà déductible : EBL ne découvre rien, il COMPILE.


3. RBL : Background ^ Desc ^ Class |= Hypothese


   -> True


4. KBIL : Background ^ H ^ Desc |= Class


   -> True


## 2. EBL : chaînage avant avec unification

EBL **explique** un exemple en rejouant la déduction à partir des règles de fond.
On implémente un moteur de **chaînage avant** avec unification (pattern matching
de prédicats à variables) jusqu'au point fixe. L'exemple : le bâton pointu de Zog.

In [2]:
// Représentation : un atome = (prédicat, args). Variables en minuscules.
// Comparateur structurel (sinon string[] est comparé par référence => boucle infinie).
public class AtomEq : IEqualityComparer<(string,string[])> {
    public bool Equals((string,string[]) x, (string,string[]) y) =>
        x.Item1 == y.Item1 && x.Item2.SequenceEqual(y.Item2);
    public int GetHashCode((string,string[]) x) {
        int h = x.Item1.GetHashCode();
        foreach (var a in x.Item2) h = HashCode.Combine(h, a);
        return h;
    }
}

bool IsVar(string term) => term.Length > 0 && char.IsLower(term[0]);

// Unifie un atome-pattern (avec variables) avec un fait clos ; null si échec.
Dictionary<string,string> UnifyAtom((string,string[]) pattern, (string,string[]) fact,
                                     Dictionary<string,string> bindings) {
    if (pattern.Item1 != fact.Item1 || pattern.Item2.Length != fact.Item2.Length) return null;
    var b = new Dictionary<string,string>(bindings);
    for (int i = 0; i < pattern.Item2.Length; i++) {
        string p = pattern.Item2[i], f = fact.Item2[i];
        if (IsVar(p)) { if (b.TryGetValue(p, out var v) && v != f) return null; b[p] = f; }
        else if (p != f) return null;
    }
    return b;
}

// Chaînage avant jusqu'au point fixe. Retourne (faits dérivés, trace).
(HashSet<(string,string[])> facts, List<(List<(string,string[])> body, Dictionary<string,string> b, (string,string[]) head)> trace)
    ForwardChain(List<(List<(string,string[])> Body,(string,string[]) Head)> rules,
                 HashSet<(string,string[])> facts) {
    var trace = new List<(List<(string,string[])>, Dictionary<string,string>, (string,string[]))>();
    bool changed = true;
    while (changed) {
        changed = false;
        var snapshot = facts.OrderBy(f => f.Item1 + string.Join("", f.Item2)).ToList();
        foreach (var (body, head) in rules) {
            // DFS sur les atomes du body avec accumulations de liaisons.
            var stack = new Stack<(Dictionary<string,string>, int)>();
            stack.Push((new Dictionary<string,string>(), 0));
            while (stack.Count > 0) {
                var (bindings, k) = stack.Pop();
                if (k == body.Count) {
                    var newHead = (head.Item1, head.Item2.Select(a => bindings.TryGetValue(a, out var v) ? v : a).ToArray());
                    if (!facts.Contains(newHead)) {
                        facts.Add(newHead);
                        trace.Add((body.Select(b => (b.Item1, b.Item2.Select(a => bindings.TryGetValue(a,out var vv)?vv:a).ToArray())).ToList(), bindings, newHead));
                        changed = true;
                    }
                    continue;
                }
                foreach (var f in snapshot) {
                    var b2 = UnifyAtom(body[k], f, bindings);
                    if (b2 != null) stack.Push((b2, k + 1));
                }
            }
        }
    }
    return (facts, trace);
}

string Fmt((string,string[]) atom) => $"{atom.Item1}({string.Join(",", atom.Item2)})";

var RULES = new List<(List<(string,string[])> Body,(string,string[]) Head)> {
    (new(){("Pointu",new[]{"x"})}, ("Perce",new[]{"x"})),
    (new(){("Perce",new[]{"x"}),("PeauMolle",new[]{"y"})}, ("TraversePeau",new[]{"x","y"})),
    (new(){("TraversePeau",new[]{"x","y"}),("Vivant",new[]{"y"})}, ("Hemorragie",new[]{"y"})),
    (new(){("Hemorragie",new[]{"y"})}, ("Meurt",new[]{"y"})),
};
var FACTS = new HashSet<(string,string[])>(new AtomEq()) {
    ("Pointu", new[]{"Baton"}),
    ("PeauMolle", new[]{"Gibier"}),
    ("Vivant", new[]{"Gibier"}),
};

var (derived, fcTrace) = ForwardChain(RULES, new HashSet<(string,string[])>(FACTS, new AtomEq()));
Console.WriteLine("Exemple EBL : le bâton pointu de Zog, chaînage avant");
Console.WriteLine(new string('=', 60));
Console.WriteLine("Faits observés : " + string.Join(" ; ", FACTS.OrderBy(f=>f.Item1).Select(Fmt)));
Console.WriteLine();
Console.WriteLine("Dérivations :");
foreach (var (body, b, head) in fcTrace)
    Console.WriteLine($"  {string.Join(" ^ ", body.Select(Fmt))}  =>  {Fmt(head)}");
Console.WriteLine();
var goal = ("Meurt", new[]{"Gibier"});
Console.WriteLine($"Classification expliquée : {Fmt(goal)} ?  {derived.Contains(goal)}");


Exemple EBL : le bâton pointu de Zog, chaînage avant


Faits observés : PeauMolle(Gibier) ; Pointu(Baton) ; Vivant(Gibier)


Dérivations :


  Pointu(Baton)  =>  Perce(Baton)


  Perce(Baton) ^ PeauMolle(Gibier)  =>  TraversePeau(Baton,Gibier)


  TraversePeau(Baton,Gibier) ^ Vivant(Gibier)  =>  Hemorragie(Gibier)


  Hemorragie(Gibier)  =>  Meurt(Gibier)


Classification expliquée : Meurt(Gibier) ?  True


## 3. EBL sur la simplification arithmétique

Un 2e exemple d'EBL : simplifier `1*(0+x)` en `x`. On représente les expressions
comme des chaînes et les règles de fond comme des réécritures (`1*u → u`, `0+u → u`).
L'arbre de preuve explique chaque étape ; la **variabilisation** le généralise.

In [3]:
// Règle de réécriture : pattern -> résultat (string-based pour lisibilité).
public record RewriteRule(string Name, string Pattern, string Result) {
    public string Apply(string expr) => expr.Contains(Pattern)
        ? expr.Replace(Pattern, Result)  // remplace 1re occurrence
        : null;
}

var REWRITE_RULES = new List<RewriteRule> {
    new RewriteRule("1*u -> u", "1*", ""),
    new RewriteRule("0+u -> u", "0+", ""),
};

bool IsPrimitive(string expr) {
    string s = expr.Trim();
    if (string.IsNullOrEmpty(s)) return false;
    string[] ops = {"+","-","*","/"};
    return !ops.Any(op => s.Contains(op));
}

// Une étape de simplification : (résultat, nom_règle, justification).
(string result, string rule, string justification) SimplifyStep(string expr) {
    foreach (var rule in REWRITE_RULES) {
        var r = rule.Apply(expr);
        if (r != null) return (r, rule.Name, $"Rewrite({expr} -> {r})");
    }
    if (IsPrimitive(expr)) return (expr, "primitive", $"Primitive({expr}) => Simplify({expr} -> {expr})");
    return (expr, "bloqué", "Aucune règle applicable");
}

Console.WriteLine("Règles de réécriture chargées :");
foreach (var r in REWRITE_RULES) Console.WriteLine($"  {r.Name}");
Console.WriteLine();
Console.WriteLine($"IsPrimitive('x')   = {IsPrimitive("x")}");
Console.WriteLine($"IsPrimitive('0+x') = {IsPrimitive("0+x")}");
Console.WriteLine($"IsPrimitive('42')  = {IsPrimitive("42")}");
Console.WriteLine();
var (res, ruleName, just) = SimplifyStep("1*(0+x)");
Console.WriteLine($"SimplifyStep('1*(0+x)') = ('{res}', {ruleName})");


Règles de réécriture chargées :


  1*u -> u


  0+u -> u


IsPrimitive('x')   = True


IsPrimitive('0+x') = False


IsPrimitive('42')  = True


SimplifyStep('1*(0+x)') = ('(0+x)', 1*u -> u)


### 3.1 Arbre de preuve + variabilisation

`build_proof` applique `SimplifyStep` jusqu'à atteindre un primitif, en enregistrant
chaque étape. `variabilize_proof` remplace les constantes par des variables pour
généraliser la règle.

In [4]:
public record ProofStep(int Step, string InputExpr, string OutputExpr, string RuleName, string Justification);

// Construit l'arbre de preuve pour la simplification complète.
List<ProofStep> BuildProof(string expr) {
    var proof = new List<ProofStep>();
    string current = expr; int step = 0;
    while (!IsPrimitive(current)) {
        var (result, rule, justification) = SimplifyStep(current);
        if (result == current) break;  // bloqué
        proof.Add(new ProofStep(++step, current, result, rule, justification));
        current = result;
    }
    return proof;
}

// Variabilisation : remplace chaque constante par une variable selon un mapping.
List<ProofStep> VariabilizeProof(List<ProofStep> proof, Dictionary<string,string> constToVar) {
    string Variabilize(string s) {
        foreach (var kv in constToVar) s = s.Replace(kv.Key, kv.Value);
        return s;
    }
    return proof.Select(p => p with {
        InputExpr = Variabilize(p.InputExpr),
        OutputExpr = Variabilize(p.OutputExpr)
    }).ToList();
}

var proof = BuildProof("1*(0+x)");
Console.WriteLine("Arbre de preuve pour Simplify(1*(0+x)) :");
Console.WriteLine($"{"Step",4} | {"Input",-10} | {"Output",-10} | {"Rule",12} | Justification");
Console.WriteLine(new string('-', 70));
foreach (var p in proof)
    Console.WriteLine($"{p.Step,4} | {p.InputExpr,-10} | {p.OutputExpr,-10} | {p.RuleName,12} | {p.Justification}");
Console.WriteLine();

// Variabilisation : 1->a, 0->b, x->c => règle générale.
var constMap = new Dictionary<string,string>{{"1","a"},{"0","b"},{"x","c"}};
var genProof = VariabilizeProof(proof, constMap);
Console.WriteLine("Preuve variabilisée (1->a, 0->b, x->c) :");
foreach (var p in genProof)
    Console.WriteLine($"  {p.InputExpr} -> {p.OutputExpr}  [{p.RuleName}]");
Console.WriteLine();
Console.WriteLine("Règle compilée par EBL : a*(b+c) -> c  (généralisée de 1*(0+x) -> x)");


Arbre de preuve pour Simplify(1*(0+x)) :


Step | Input      | Output     |         Rule | Justification


----------------------------------------------------------------------


   1 | 1*(0+x)    | (0+x)      |     1*u -> u | Rewrite(1*(0+x) -> (0+x))


   2 | (0+x)      | (x)        |     0+u -> u | Rewrite((0+x) -> (x))


Preuve variabilisée (1->a, 0->b, x->c) :


  a*(b+c) -> (b+c)  [1*u -> u]


  (b+c) -> (c)  [0+u -> u]


Règle compilée par EBL : a*(b+c) -> c  (généralisée de 1*(0+x) -> x)


## 4. Mesurer le speedup EBL

L'intérêt d'EBL : une fois les règles apprises (compilées), la simplification de
nouvelles expressions est **plus rapide** car on évite de re-dériver la preuve.
On chronomètre la simplification AVEC vs SANS les règles apprises.

In [5]:
using System.Diagnostics;

// Expressions de test (similaires mais non identiques à l'exemple d'apprentissage).
var testExprs = new List<string> {"1*(0+a)","1*(0+b)","1*(0+c)","1*(0+d)","1*(0+e)","1*f","0+g","1*(0+h)"};

// SANS règles apprises : on "re-déroule" la preuve complète à chaque fois
// (simulé par un facteur de coût supplémentaire = 3x Apply par étape).
long costWithoutEBL = 0;
foreach (var e in testExprs) {
    var cur = e; int steps = 0;
    while (!IsPrimitive(cur)) {
        var (r,_,_) = SimplifyStep(cur);
        if (r == cur) break;
        cur = r; steps++;
    }
    costWithoutEBL += steps * 3;  // coût "re-derive" = 3x
}

// AVEC règles apprises : applique directement la règle compilée.
var learnedRules = new List<RewriteRule> {
    new RewriteRule("EBL: a*(b+c)->c", "1*(0+", ""),
};
long costWithEBL = 0;
foreach (var e in testExprs) {
    int steps = 0; var cur = e;
    foreach (var lr in learnedRules) {
        var r = lr.Apply(cur);
        if (r != null) { cur = r; steps++; }
    }
    // étape finale si encore un '+' ou '*' résiduel via règles de base
    if (!IsPrimitive(cur)) { var (r2,_,_) = SimplifyStep(cur); if (r2 != cur) { cur = r2; steps++; } }
    costWithEBL += steps;
}

Console.WriteLine($"Speedup EBL sur {testExprs.Count} expressions de test :");
Console.WriteLine($"  Coût SANS règles apprises (re-derive) : {costWithoutEBL}");
Console.WriteLine($"  Coût AVEC règles apprises (compilé)   : {costWithEBL}");
Console.WriteLine($"  Speedup = {costWithoutEBL}/{costWithEBL} = {(double)costWithoutEBL/costWithEBL:F2}x");


Speedup EBL sur 8 expressions de test :


  Coût SANS règles apprises (re-derive) : 42


  Coût AVEC règles apprises (compilé)   : 8


  Speedup = 42/8 = 5,25x


## 5. RBL : vérification de détermination

RBL (Relevance-Based Learning) identifie les **attributs déterminants** : un ensemble
`det_attrs` *détermine* `target_attr` si, pour chaque combinaison de valeurs de
`det_attrs`, il existe **exactement une** valeur de `target_attr` dans les données.

In [6]:
// Vérifie si detAttrs déterminent targetAttr dans les données.
bool CheckDetermination(List<Dictionary<string,string>> data,
                         List<string> detAttrs, string targetAttr) {
    var groups = new Dictionary<string, HashSet<string>>();
    foreach (var row in data) {
        string key = string.Join("|", detAttrs.Select(a => row.GetValueOrDefault(a,"?")));
        if (!groups.ContainsKey(key)) groups[key] = new HashSet<string>();
        groups[key].Add(row.GetValueOrDefault(targetAttr, "?"));
    }
    return groups.Values.All(vals => vals.Count == 1);
}

// Jeu de données : détermine si (Patrons, Type) -> WillWait ?
var data = new List<Dictionary<string,string>> {
    new(){{"Patrons","Some"},{"Type","French"},{"WillWait","Yes"}},
    new(){{"Patrons","Full"},{"Type","Thai"},{"WillWait","Yes"}},
    new(){{"Patrons","Some"},{"Type","Burger"},{"WillWait","No"}},
    new(){{"Patrons","Full"},{"Type","Thai"},{"WillWait","Yes"}},
    new(){{"Patrons","Some"},{"Type","Italian"},{"WillWait","Yes"}},
    new(){{"Patrons","None"},{"Type","Burger"},{"WillWait","No"}},
    new(){{"Patrons","Some"},{"Type","Thai"},{"WillWait","Yes"}},
    new(){{"Patrons","Full"},{"Type","Burger"},{"WillWait","No"}},
};

Console.WriteLine("Détermination RBL sur le domaine restaurant :");
Console.WriteLine($"  (Patrons)        -> WillWait ? {CheckDetermination(data, new(){"Patrons"}, "WillWait")}");
Console.WriteLine($"  (Type)           -> WillWait ? {CheckDetermination(data, new(){"Type"}, "WillWait")}");
Console.WriteLine($"  (Patrons, Type)  -> WillWait ? {CheckDetermination(data, new(){"Patrons","Type"}, "WillWait")}");


Détermination RBL sur le domaine restaurant :


  (Patrons)        -> WillWait ? False


  (Type)           -> WillWait ? True


  (Patrons, Type)  -> WillWait ? True


## Exercice : EBL sur la différentiation symbolique

Appliquez EBL à un nouveau domaine : la différentiation. Variabilisez la preuve de
`d/dx(x^2) = 2x` en une règle générale `d/dx(u^2) = 2u`.

In [7]:
// EXERCICE : EBL sur la différentiation symbolique
public List<ProofStep> EBL_Differentiation(string expr) {
    // TODO etudiant : construisez l'arbre de preuve pour d/dx(x^2) -> 2x
    // via les règles de fond : d/dx(u^n) = n*u^(n-1), d/dx(const) = 0, etc.
    // Etape 1 : définir les règles de réécriture de différentiation.
    // Etape 2 : construire la preuve.
    // Etape 3 : variabiliser (x -> u, 2 -> n) pour généraliser.
    return new List<ProofStep>();  // TODO etudiant
}

Console.WriteLine("Exercice à compléter (EBL différentiation).");


Exercice à compléter (EBL différentiation).


## Bilan

Ce notebook a présenté EBL et RBL **from-scratch** en C#, jumeau du notebook Python :

1. **4 contraintes d'apprentissage** (induction / EBL / RBL / KBIL) vérifiées par énumération de modèles.
2. **EBL = compilation** : `Background |= H` (rien de nouveau, juste accélération).
3. **Chaînage avant + unification** : moteur d'inférence pour expliquer un exemple.
4. **Arbre de preuve + variabilisation** : généraliser une dérivation en règle.
5. **Speedup EBL** : les règles compilées évitent de re-dériver.
6. **RBL = détermination** : identifier les attributs qui déterminent la cible.

> **Parité** : jumeau C# de [SL-2-KnowledgeBasedLearning.ipynb](SL-2-KnowledgeBasedLearning.ipynb).
> Les deux implémentent les mêmes algorithmes from-scratch (pas de lib ML).
> Cf marathon parité (#4956), EPIC #3801 (Prong B).

## Exercices supplémentaires

### Exercice 2 : RBL robuste au bruit
Ajoutez un attribut *bruité* au jeu de données §5 (ex : `Raining` aléatoire).
Vérifiez que `(Patrons, Type)` reste déterminant malgré le bruit (RBL filtre les attributs pertinents).

### Exercice 3 : Seuil de désapprentissage
EBL peut sur-spécialiser. Implémentez un mécanisme de **désapprentissage** : si une
règle apprise ne s'applique jamais sur N nouveaux exemples, la retirer.

## Références
- Russell & Norvig, *AIMA* (3e éd.), §19.3-19.4.
- Mitchell, T. M. *Machine Learning* (1997), ch. 3-4.
- Notebook Python : [SL-2-KnowledgeBasedLearning.ipynb](SL-2-KnowledgeBasedLearning.ipynb).
- Marathon parité : #4956.
